In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import glob
import os
from pathlib import Path
from multiprocessing import Pool, cpu_count
from typing import List
from numba import njit
import re
import math

In [6]:
def process_file(file: str) -> pd.DataFrame:
    file_path = Path(file)
    parts = file_path.name.split("_")
    chromosome = "".join(parts[1:3])  # e.g., ['chr10', '1'] → 'chr101'

    df = pd.read_csv(file_path, sep="\t", header=None)
    df.insert(0, "CHROM", chromosome)
    df.rename(columns={0: "START", 1: "END", 2: "RHO"}, inplace=True)
    return df


def read_files_parallel(files: str) -> List[pd.DataFrame]:
    file_names = glob.glob(files, recursive=True)
    with Pool(processes=cpu_count()//2) as pool:
        dfs = pool.map(process_file, file_names)
    return dfs

recombination_rate = read_files_parallel("../output/n50_*_W100_P20.rmap")


In [7]:
def make_windows(data_frame: pd.DataFrame, w_size: int) -> pd.DataFrame:
    raw_data = data_frame.to_numpy()
    midpoints = (raw_data[:, 1].astype(int) + raw_data[:, 2].astype(int)) // 2
    bin_ids = midpoints // w_size
    
    unique_bins, inverse_indeces = np.unique(bin_ids, return_inverse=True)
    binned_sums = np.zeros(len(unique_bins))
    binned_count = np.zeros(len(unique_bins))
    
    np.add.at(binned_sums, inverse_indeces, raw_data[:, 3].astype(float))
    np.add.at(binned_count, inverse_indeces, 1)
    
    binned_means = (binned_sums / binned_count) * 100 * 1e6
    starts = unique_bins * w_size
    ends = starts + w_size
    midpoint = (starts + ends) // 2
    chrom = raw_data[0, 0]
    results = np.column_stack([
        np.full(len(unique_bins), chrom),
        starts,
        ends,
        binned_means,
        midpoint
    ])
    dataframe = pd.DataFrame(results, columns=["chrom", "start", "end", "cM/Mb", "midpoint"])
    return dataframe


windows_5kb = [make_windows(df, 5000) for df in recombination_rate]
        

In [ ]:
def summarize_recombination_windows(windows: list[pd.DataFrame]) -> pd.DataFrame:
    summary_table = pd.DataFrame([
        {
            "chrom": df["chrom"].iloc[0],
            **pd.to_numeric(df["cM/Mb"], errors="coerce").describe()
        }
        for df in windows
    ])

    # Reorder and round columns
    columns = ["chrom", "count", "mean", "std", "min", "25%", "50%", "75%", "max"]
    summary_table = summary_table[columns]
    summary_table[columns[2:]] = summary_table[columns[2:]].round(3)

    # Sort chromosomes numerically if named chr1, chr2, ..., chrX, etc.
    summary_table = summary_table.sort_values(
        "chrom", 
        key=lambda x: x.str.extract(r'chr(\d+|W|Z)')[0].map(lambda s: int(s) if s and s.isdigit() else float('inf'))
    )

    return summary_table
summary_table = summarize_recombination_windows(windows_5kb)
summary_table.to_latex("recombination_rate_5kb_summary.tex", index=False, float_format="%.3f")

In [8]:
def chr_sort_key(df):
    # Ensure we're getting a string from the chrom column
    chrom = str(df.iloc[0]["chrom"]) if "chrom" in df.columns else str(df.iloc[0, 0])
    match = re.match(r"chr(\d+)", chrom)
    if match:
        return int(match.group(1))  # Numeric chromosomes
    else:
        return float("inf")         # Push non-numeric chromosomes (e.g., chrX) to the end

windows_5kb.sort(key=chr_sort_key)

In [36]:
def plot_recombination_rate(windows: List[pd.DataFrame], plot_name: str) -> None:
    n = len(windows)
    cols = 4
    rows = math.ceil(n/cols)
     
    figure = make_subplots(rows=rows, 
                      cols=cols, 
                      subplot_titles=[df["chrom"].iloc[0] for df in windows], 
                      shared_yaxes=True)
    for i, df in enumerate(windows):
        row = i // cols + 1
        col = i % cols + 1
        
        figure.add_trace(
            go.Line(x=df["midpoint"].astype(int), y=df["cM/Mb"].astype(float), name=df["chrom"].iloc[0]),
            row,
            col
        )
        figure.update_layout(
        height=300 * rows,
        width=300 * cols,
        title = {
            'text':'Recombination Rate over Genomic Positions',
            'x': 0.5,
            'y': 0.99,
            'yanchor': 'top',
            'xanchor': 'center',
            'font': dict(size=26)
            },
        showlegend=False,
        margin=dict(t=80, l=20, r=20, b=20)
    )

    figure.update_xaxes(title_text="Genomic Position")
    figure.update_yaxes(title_text="cM/Mb")
    
    if plot_name:
        figure.write_image(plot_name)
    figure.show()
    


plot_recombination_rate(windows_5kb, 'recombination_rate_50_indv_5kb.pdf')
        